# Detector de plágio com similaridade de Jaccard
Este notebook replica a abordagem quantitativa da atividade TC.4 para montar um detector simples de plágio usando vocabulário tokenizado com NLTK.

## Fluxo geral
1. Instalar/carregar dependências (NLTK).
2. Tokenizar textos em português e montar o vocabulário de cada documento.
3. Calcular a similaridade de Jaccard entre os vocabulários (interseção/união).
4. Classificar como plágio quando o índice exceder 50%.
5. Validar o programa com cenários positivos e negativos.

In [1]:
# Dependências principais
from __future__ import annotations

from pathlib import Path
from typing import Dict, Iterable, Set, Tuple

import nltk

for resource in ("punkt",):
    try:
        nltk.data.find(f"tokenizers/{resource}")
    except LookupError:
        nltk.download(resource)

print("Recursos do NLTK prontos.")

Recursos do NLTK prontos.


In [2]:
# Funções de tokenização, cálculo de Jaccard e decisão de plágio
def tokenize(texto: str) -> Iterable[str]:
    return (
        token.lower()
        for token in nltk.word_tokenize(texto, language="portuguese")
        if token.isalpha()
    )

def vocabulario(texto: str) -> Set[str]:
    return set(tokenize(texto))

def similaridade_jaccard(doc_a: str, doc_b: str) -> Tuple[float, Set[str], Set[str]]:
    vocab_a = vocabulario(doc_a)
    vocab_b = vocabulario(doc_b)
    uniao = vocab_a | vocab_b
    intersecao = vocab_a & vocab_b
    indice = len(intersecao) / len(uniao) if uniao else 0.0
    return indice, vocab_a, vocab_b

def detectar_plagio(doc_referencia: str, doc_suspeito: str, limiar: float = 0.5) -> Dict[str, float | int | str]:
    indice, vocab_a, vocab_b = similaridade_jaccard(doc_referencia, doc_suspeito)
    resultado = "PLÁGIO" if indice > limiar else "OK"
    return {
        "indice": indice,
        "resultado": resultado,
        "limiar": limiar,
        "vocab_referencia": len(vocab_a),
        "vocab_suspeito": len(vocab_b),
        "tokens_comuns": len(vocab_a & vocab_b),
        "tamanho_uniao": len(vocab_a | vocab_b),
    }

def imprimir_relatorio(titulo: str, relatorio: Dict[str, float | int | str]) -> None:
    print(titulo)
    print(
        f"  Similaridade de Jaccard: {relatorio['indice']*100:.2f}% -> {relatorio['resultado']} ",
        f"(limiar {relatorio['limiar']*100:.0f}%)"
    )
    print(
        f"  Vocabulários: {relatorio['vocab_referencia']} vs {relatorio['vocab_suspeito']} ",
        f"| Interseção: {relatorio['tokens_comuns']} | União: {relatorio['tamanho_uniao']}"
    )
    print()

## Casos de teste
- *Caso 1:* documento suspeito reutiliza praticamente todo o conteúdo de referência (deve ser marcado como plágio).
- *Caso 2:* documento autoral fala de outro tema e deve ficar abaixo do limiar.

In [4]:
referencia = (
    "O pesquisador descreveu meticulosamente o bairro, suas vielas, o comércio "
    "popular e as mudanças sociais percebidas ao longo de décadas."
)
suspeito_plagio = referencia + " O mesmo relato foi copiado integralmente, sem citar a fonte."
suspeito_autoral = (
    "Uma crônica rural narra as colheitas, festas populares e memórias de infância "
    "vividas no interior nordestino."
)

relatorio_plagio = detectar_plagio(referencia, suspeito_plagio)
imprimir_relatorio("Caso suspeito", relatorio_plagio)

relatorio_autoral = detectar_plagio(referencia, suspeito_autoral)
imprimir_relatorio("Caso autoral", relatorio_autoral)

assert relatorio_plagio["resultado"] == "PLÁGIO", "Era esperado detectar plágio no caso 1"
assert relatorio_autoral["resultado"] == "OK", "Era esperado liberar o caso 2"
print("Todos os testes passaram.")

Caso suspeito
  Similaridade de Jaccard: 66.67% -> PLÁGIO  (limiar 50%)
  Vocabulários: 18 vs 27  | Interseção: 18 | União: 27

Caso autoral
  Similaridade de Jaccard: 9.68% -> OK  (limiar 50%)
  Vocabulários: 18 vs 16  | Interseção: 3 | União: 31

Todos os testes passaram.


## Como usar com arquivos próprios
Preencha os caminhos abaixo com textos externos para reutilizar o detector nas suas avaliações.

In [ ]:
# Exemplo opcional para execução manual
# Substitua pelos arquivos desejados e descomente o bloco.
# caminho_referencia = Path("dados/referencia.txt")
# caminho_suspeito = Path("dados/suspeito.txt")
# texto_referencia = caminho_referencia.read_text(encoding="utf-8")
# texto_suspeito = caminho_suspeito.read_text(encoding="utf-8")
# relatorio_custom = detectar_plagio(texto_referencia, texto_suspeito, limiar=0.5)
# imprimir_relatorio(f"Comparação: {caminho_referencia.name} x {caminho_suspeito.name}", relatorio_custom)